# Synth-Scenario Tabellen updaten: Kurzlabels + Benchmark-Zeile (CSV + LaTeX)

Dieses Notebook nimmt deine bestehenden DRL-Tabellen aus:
`.../runs/synth_final/_tables/synth_table_<scenario>.csv`

und macht:
1) `config_label` → Kurzlabels (LOG-S0, ICVaR-S0, ICVaR+DD-S0, ...)
2) optional: Benchmark-Zeile aus einer `metrics_per_scenario.csv` ergänzen
3) schreibt neue CSV + LaTeX-Tabellen (\input-fertig).


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

# ====== INPUT ======
TABLE_DIR = Path(r"C:/Dev/Bachelorarbeit/results/accounting/runs/synth_final/_tables")

# Optional: Benchmark metrics_per_scenario.csv
# Beispiel (wenn du alles in EINER metrics_per_scenario.csv sammelst):
# BENCH_METRICS = Path(r"C:/Dev/Bachelorarbeit/results/accounting/runs/synth_final/bench_synth/_report/metrics_per_scenario.csv")
# Oder: Liste mehrerer Files (falls du pro Szenario separat speicherst)
BENCH_METRICS = BENCH_METRICS = sorted(Path(r"C:/Dev/Bachelorarbeit/data/benchmark_fin_per_year/scenarios").rglob("metrics_per_scenario.csv")) # Path(...) oder [Path(...), Path(...)]
BENCH_LABEL = "Benchmark"

# ====== OUTPUT ======
# Wir überschreiben NICHT, sondern schreiben *_final.*
OUT_DIR = TABLE_DIR

# Optional: direkt ins LaTeX-Projekt schreiben
LATEX_TABLE_DIR = Path(r"C:/Users/Dr. LongRodeo/Documents/Studium/8. Semester Bacherlorarbeit/latex-ba/tables/scenarios")
LATEX_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("TABLE_DIR:", TABLE_DIR)
print("LATEX_TABLE_DIR:", LATEX_TABLE_DIR)


TABLE_DIR: C:\Dev\Bachelorarbeit\results\accounting\runs\synth_final\_tables
LATEX_TABLE_DIR: C:\Users\Dr. LongRodeo\Documents\Studium\8. Semester Bacherlorarbeit\latex-ba\tables\scenarios


In [8]:
# ====== Kurzlabel-Mapping ======
LABEL_MAP = {
    "log_ppo_S0": "LOG-S0",
    "log_ppo_S1": "LOG-S1",
    "icvar_ppo_S0": "ICVaR-S0",
    "icvar_ppo_S1": "ICVaR-S1",
    "icvar_dd_ppo_S0": "ICVaR+DD-S0",
    "icvar_dd_ppo_S1": "ICVaR+DD-S1",
}

def shorten_label(s: str) -> str:
    s = str(s)
    for key, lab in LABEL_MAP.items():
        if key in s:
            return lab
    return s


In [9]:
SCENARIOS = [
    "bear_1y",
    "side_lowvol_1y",
    "side_highvol_1y",
]

tables = {}
for sc in SCENARIOS:
    p = TABLE_DIR / f"synth_table_{sc}.csv"
    df = pd.read_csv(p)
    df["config_label"] = df["config_label"].map(shorten_label)
    tables[sc] = df

tables["bear_1y"].head()


,config_label,total_cum_return,total_maxdd,ex_cum_return,ex_sharpe,ex_sortino,ex_cvar_95,ex_calmar,avg_cost_rate,avg_turnover,psr_ex,dsr_ex
0,LOG-S1,0.512178,-0.705288,0.454243,0.942602,1.549893,-0.174571,0.642086,0.000010,0.250572,NaN,NaN
1,ICVaR-S0,0.179442,-0.309034,0.134255,0.515910,0.764703,-0.053919,0.418670,0.000022,0.087013,NaN,NaN
2,ICVaR-S1,0.028252,-0.319109,-0.011142,0.147180,0.200557,-0.054462,-0.033608,0.000020,0.096419,NaN,NaN
3,LOG-S0,-0.012285,-0.839666,-0.050126,0.727981,1.131522,-0.198803,-0.059715,0.000048,0.099294,NaN,NaN
4,ICVaR+DD-S1,-0.047206,-0.186880,-0.083709,-0.339740,-0.445114,-0.032714,-0.420014,0.000011,0.158324,NaN,NaN


In [10]:
def load_benchmark_metrics(bench_metrics):
    if bench_metrics is None:
        return None
    if isinstance(bench_metrics, (list, tuple)):
        dfs = [pd.read_csv(p) for p in bench_metrics]
        bm = pd.concat(dfs, ignore_index=True)
    else:
        bm = pd.read_csv(bench_metrics)
    return bm

bm = load_benchmark_metrics(BENCH_METRICS)
bm.head() if bm is not None else "BENCH_METRICS=None (überspringe Benchmark-Integration)"

if bm is not None and bm["scenario"].eq("full").all():
    bm["scenario"] = bm["run"].str.replace(r"^bench_", "", regex=True)
    bm["test_dir"] = bm["scenario"]



In [11]:
# ====== Benchmark -> Zeilen pro Szenario bauen (falls vorhanden) ======
def bench_rows_per_scenario(bm: pd.DataFrame) -> dict[str, dict]:
    # Erwartete Spalten wie in metrics_per_scenario.csv
    keep = [
        "total_cum_return","total_maxdd","ex_cum_return","ex_sharpe","ex_sortino",
        "ex_cvar_95","ex_calmar","avg_cost_rate","avg_turnover","psr_ex","dsr_ex"
    ]
    for c in keep + ["scenario"]:
        if c not in bm.columns:
            raise ValueError(f"Benchmark metrics fehlt Spalte: {c}")

    # Falls mehrere folds vorhanden sind: Mittelwert
    agg = bm.groupby("scenario", as_index=False)[keep].mean()

    out = {}
    for _, r in agg.iterrows():
        sc = str(r["scenario"])
        row = {"config_label": BENCH_LABEL}
        for c in keep:
            row[c] = float(r[c]) if pd.notna(r[c]) else np.nan
        out[sc] = row
    return out

bench_rows = bench_rows_per_scenario(bm) if bm is not None else {}
bench_rows


{'bear_1y': {'config_label': 'Benchmark',
  'total_cum_return': 0.0175995367849877,
  'total_maxdd': -0.0611257239459812,
  'ex_cum_return': -0.0212342885626729,
  'ex_sharpe': -0.3591682417850272,
  'ex_sortino': -0.4482014864166246,
  'ex_cvar_95': -0.0089908351943636,
  'ex_calmar': -0.2687202878095538,
  'avg_cost_rate': 1.3167376462817757e-06,
  'avg_turnover': 0.1510109974907156,
  'psr_ex': nan,
  'dsr_ex': nan},
 'side_highvol_1y': {'config_label': 'Benchmark',
  'total_cum_return': 0.0877203540804738,
  'total_maxdd': -0.0742105688312079,
  'ex_cum_return': 0.0462105648849036,
  'ex_sharpe': 0.4724210501298356,
  'ex_sortino': 0.7306808187526488,
  'ex_cvar_95': -0.0144858876888487,
  'ex_calmar': 0.5107482945256265,
  'avg_cost_rate': 1.304404796427566e-06,
  'avg_turnover': 0.1495965957960071,
  'psr_ex': nan,
  'dsr_ex': nan},
 'side_lowvol_1y': {'config_label': 'Benchmark',
  'total_cum_return': 0.0352841608606375,
  'total_maxdd': -0.0207632704756138,
  'ex_cum_return': -

In [12]:
# ====== Tabellen updaten + schreiben ======
KEEP_COLS = [
    "config_label",
    "total_cum_return","total_maxdd",
    "ex_cum_return","ex_sharpe","ex_sortino","ex_cvar_95","ex_calmar",
    "avg_cost_rate","avg_turnover",
    "psr_ex","dsr_ex",
]

def make_tex_tabular(df: pd.DataFrame) -> str:
    # schöner: NaN -> --, 4 decimals
    return df.to_latex(
        index=False,
        float_format="%.4f",
        na_rep="--",
        escape=False,
    )

for sc in SCENARIOS:
    df = tables[sc][KEEP_COLS].copy()

    # Benchmark-Zeile ergänzen (falls vorhanden)
    if sc in bench_rows:
        b = pd.DataFrame([bench_rows[sc]])[KEEP_COLS]
        df = pd.concat([df, b], ignore_index=True)

    # Sortieren: optional nach ex_calmar (absteigend)
    df = df.sort_values("ex_calmar", ascending=False, na_position="last")

    # Output
    out_csv = OUT_DIR / f"synth_table_{sc}_final.csv"
    out_tex = OUT_DIR / f"synth_table_{sc}_final.tex"

    df.to_csv(out_csv, index=False)
    out_tex.write_text(make_tex_tabular(df), encoding="utf-8")

    # Copy ins LaTeX-Projekt
    (LATEX_TABLE_DIR / out_tex.name).write_text(out_tex.read_text(encoding="utf-8"), encoding="utf-8")

    print("Wrote:", out_csv)
    print("Wrote:", out_tex)
    print("Copied to:", LATEX_TABLE_DIR / out_tex.name)


Wrote: C:\Dev\Bachelorarbeit\results\accounting\runs\synth_final\_tables\synth_table_bear_1y_final.csv
Wrote: C:\Dev\Bachelorarbeit\results\accounting\runs\synth_final\_tables\synth_table_bear_1y_final.tex
Copied to: C:\Users\Dr. LongRodeo\Documents\Studium\8. Semester Bacherlorarbeit\latex-ba\tables\scenarios\synth_table_bear_1y_final.tex
Wrote: C:\Dev\Bachelorarbeit\results\accounting\runs\synth_final\_tables\synth_table_side_lowvol_1y_final.csv
Wrote: C:\Dev\Bachelorarbeit\results\accounting\runs\synth_final\_tables\synth_table_side_lowvol_1y_final.tex
Copied to: C:\Users\Dr. LongRodeo\Documents\Studium\8. Semester Bacherlorarbeit\latex-ba\tables\scenarios\synth_table_side_lowvol_1y_final.tex
Wrote: C:\Dev\Bachelorarbeit\results\accounting\runs\synth_final\_tables\synth_table_side_highvol_1y_final.csv
Wrote: C:\Dev\Bachelorarbeit\results\accounting\runs\synth_final\_tables\synth_table_side_highvol_1y_final.tex
Copied to: C:\Users\Dr. LongRodeo\Documents\Studium\8. Semester Bacherlor

## LaTeX Einbindung

Im Ergebniskapitel:

```latex
\begin{table}[t]
\centering
\scriptsize
\setlength{\tabcolsep}{3pt}
\input{tables/scenarios/synth_table_bear_1y_final.tex}
\caption{Ergebnisse im Regime \texttt{bear\_1y}.}
\label{tab:synth_bear}
\end{table}
```
